In [1]:
import json

with open('parsed_6000.json', 'r') as f:
    parsed = json.load(f)

print(f"Loaded {len(parsed)} examples")

Loaded 6000 examples


In [2]:
COMPRESS_PROMPT = """You are a reasoning trace compressor.

Below is a LONG reasoning trace from a math problem. Your job is to rewrite it as a SHORT, 
direct solution that keeps ONLY the essential logical steps needed to reach the answer.

REMOVE:
- Self-doubt ("let me think", "hmm", "wait")
- Redundant verification ("let me check", "let me verify")
- Repeated explanations of the same idea
- Dead ends and backtracking
- Obvious steps that add no value

KEEP:
- The core logical chain from problem to answer
- Key equations and calculations
- Critical insights that lead to the solution

RULES:
- The compressed trace MUST lead to the same final answer
- Target length: {target_words} words maximum
- Write in first person, natural reasoning style
- Do NOT include the final answer — only the thinking steps

ORIGINAL TRACE:
{trace}

Write the compressed reasoning trace:"""

In [3]:
import os
from dotenv import load_dotenv
import google.genai as genai

load_dotenv()

client=genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

## TEST

In [4]:
import time

# Compress just 3 easy examples as a test
easy_examples = [p for p in parsed if p['difficulty'] == 'easy']
test_compressed = {}

for i in range(3):
    ex = easy_examples[i]
    prompt = COMPRESS_PROMPT.format(target_words=150, trace=ex['thinking'])
    response = client.models.generate_content(
    model="gemini-2.5-flash-lite", 
    contents=prompt)
    
    test_compressed[f"easy_{i}"] = response.text.strip()
    time.sleep(4.5)
    print(f"Compressed easy_{i}")

# Now simulate what the final dataset will look like
print("\n\n=== FINAL TRAINING FORMAT (3 test examples) ===\n")

for i in range(3):
    ex = easy_examples[i]
    compressed_thinking = test_compressed[f"easy_{i}"]
    
    # This is exactly how each training example will look
    training_example = {
        'problem': ex['problem'],
        'difficulty': ex['difficulty'],
        'original_thinking_words': ex['thinking_word_count'],
        'compressed_thinking_words': len(compressed_thinking.split()),
        'training_text': f"<think>\n{compressed_thinking}\n</think>\n\n{ex['answer']}"
    }
    
    print(f"{'='*60}")
    print(f"EXAMPLE {i} | {training_example['original_thinking_words']} → {training_example['compressed_thinking_words']} words")
    print(f"\nPROBLEM:\n{training_example['problem'][:200]}")
    print(f"\nFULL TRAINING TEXT (what the model learns to output):")
    print(training_example['training_text'][:500])
    print()

# Also show what a HARD example looks like (no compression)
print(f"{'='*60}")
print("HARD EXAMPLE (no compression)")
hard_ex = [p for p in parsed if p['difficulty'] == 'hard'][0]
hard_training = {
    'problem': hard_ex['problem'],
    'training_text': f"<think>\n{hard_ex['thinking']}\n</think>\n\n{hard_ex['answer']}"
}
print(f"Thinking words: {hard_ex['thinking_word_count']} (kept original)")
print(f"\nPROBLEM:\n{hard_training['problem'][:200]}")
print(f"\nFULL TRAINING TEXT (first 500 chars):")
print(hard_training['training_text'][:500])

Compressed easy_0
Compressed easy_1
Compressed easy_2


=== FINAL TRAINING FORMAT (3 test examples) ===

EXAMPLE 0 | 1385 → 86 words

PROBLEM:
Given a convex polygon \( A_{n+1} \) with \( n+1 \) sides (where \( n \geq 2 \)), partition the polygon into \( n-1 \) triangles using \( n-2 \) non-intersecting diagonals (referred to as a triangulat

FULL TRAINING TEXT (what the model learns to output):
<think>
I need to find the number of triangulations for a convex polygon with $n+1$ sides. I recall that the number of triangulations for a convex polygon with $m$ sides is given by the $(m-2)$th Catalan number, $C_{m-2}$.

In this problem, the polygon has $m = n+1$ sides. Therefore, the number of triangulations is $C_{(n+1)-2} = C_{n-1}$.

The formula for the $k$th Catalan number is $C_k = \frac{1}{k+1} \binom{2k}{k}$.
To find $C_{n-1}$, I substitute $k = n-1$ into the formula:
$C_{n-1} = \frac

EXAMPLE 1 | 701 → 131 words

PROBLEM:
The locus of the complex number \( z \) that satisfies \( 2|z

## Final Compressed Reasoning

In [5]:
import time
import json

COMPRESS_PROMPT = """You are a reasoning trace compressor.

Below is a LONG reasoning trace from a math problem. Your job is to rewrite it as a SHORT, 
direct solution that keeps ONLY the essential logical steps needed to reach the answer.

REMOVE:
- Self-doubt ("let me think", "hmm", "wait")
- Redundant verification ("let me check", "let me verify")
- Repeated explanations of the same idea
- Dead ends and backtracking
- Obvious steps that add no value

KEEP:
- The core logical chain from problem to answer
- Key equations and calculations
- Critical insights that lead to the solution

RULES:
- The compressed trace MUST lead to the same final answer
- Target length: {target_words} words maximum
- Write in first person, natural reasoning style
- Do NOT include the final answer — only the thinking steps

ORIGINAL TRACE:
{trace}

Write the compressed reasoning trace:"""

# Separate by difficulty
easy_examples = [p for p in parsed if p['difficulty'] == 'easy']
medium_examples = [p for p in parsed if p['difficulty'] == 'medium']
hard_examples = [p for p in parsed if p['difficulty'] == 'hard']

# Try to load existing progress
try:
    with open('compression_progress.json', 'r') as f:
        progress = json.load(f)
    print(f"Resuming from {len(progress)} completed compressions")
except:
    progress = {}
    print("Starting fresh")

def compress_batch(examples, target_words, label):
    for i, ex in enumerate(examples):
        key = f"{label}_{i}"
        
        # Skip if already done
        if key in progress:
            continue
        
        prompt = COMPRESS_PROMPT.format(
            target_words=target_words,
            trace=ex['thinking']
        )
        
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash-lite", contents=prompt)
        
            compressed = response.text.strip()
            progress[key] = compressed
            
            # Print progress
            orig = ex['thinking_word_count']
            comp = len(compressed.split())
            print(f"[{label}] {i+1}/{len(examples)} | {orig} → {comp} words ({orig/max(comp,1):.1f}x)")
            
        except Exception as e:
            print(f"[{label}] {i+1}/{len(examples)} | ERROR: {e}")
            # Save progress before potentially crashing
            with open('compression_progress.json', 'w') as f:
                json.dump(progress, f)
            time.sleep(10)  # Wait longer on error
            continue
        
        # Save every 50 examples
        if (i + 1) % 50 == 0:
            with open('compression_progress.json', 'w') as f:
                json.dump(progress, f)
            print(f"  → Saved progress ({len(progress)} total)")
        
        # Rate limit: ~14 requests per minute to stay under 15 RPM
        time.sleep(4.5)

# Run compression
print("=== COMPRESSING EASY (2000 examples, target: 150 words) ===")
compress_batch(easy_examples, 150, "easy")

print("\n=== COMPRESSING MEDIUM (2000 examples, target: 400 words) ===")
compress_batch(medium_examples, 400, "medium")

# Final save
with open('compression_progress.json', 'w') as f:
    json.dump(progress, f)
print(f"\nDone! {len(progress)} compressions saved")

Starting fresh
=== COMPRESSING EASY (2000 examples, target: 150 words) ===
[easy] 1/2000 | 1385 → 146 words (9.5x)
[easy] 2/2000 | 701 → 139 words (5.0x)
[easy] 3/2000 | 405 → 116 words (3.5x)
[easy] 4/2000 | 961 → 131 words (7.3x)
[easy] 5/2000 | 725 → 188 words (3.9x)
[easy] 6/2000 | 895 → 85 words (10.5x)
[easy] 7/2000 | 645 → 159 words (4.1x)
[easy] 8/2000 | 695 → 106 words (6.6x)
[easy] 9/2000 | 1454 → 184 words (7.9x)
[easy] 10/2000 | 1367 → 188 words (7.3x)
[easy] 11/2000 | 651 → 105 words (6.2x)
[easy] 12/2000 | 1582 → 175 words (9.0x)
[easy] 13/2000 | 1174 → 211 words (5.6x)
[easy] 14/2000 | 471 → 193 words (2.4x)
[easy] 15/2000 | 502 → 94 words (5.3x)
[easy] 16/2000 | 775 → 153 words (5.1x)
[easy] 17/2000 | 755 → 130 words (5.8x)
[easy] 18/2000 | 922 → 82 words (11.2x)
[easy] 19/2000 | 1034 → 163 words (6.3x)
[easy] 20/2000 | 1430 → 112 words (12.8x)
[easy] 21/2000 | 434 → 122 words (3.6x)
[easy] 22/2000 | 1301 → 226 words (5.8x)
[easy] 23/2000 | 743 → 194 words (3.8x)
[easy]

In [7]:
easy_done = [k for k in progress.keys() if k.startswith('easy_')]
medium_done = [k for k in progress.keys() if k.startswith('medium_')]

print(f"Easy completed: {len(easy_done)} / 2000")
print(f"Medium completed: {len(medium_done)} / 2000")
print(f"Total: {len(progress)}")

# Find gaps
easy_indices = set(int(k.split('_')[1]) for k in easy_done)
missing_easy = [i for i in range(2000) if i not in easy_indices]
print(f"\nMissing easy indices: {len(missing_easy)}")
if len(missing_easy) < 20:
    print(f"  {missing_easy}")

Easy completed: 1993 / 2000
Medium completed: 2000 / 2000
Total: 3993

Missing easy indices: 7
  [502, 539, 764, 1171, 1753, 1756, 1909]


## Final Dataset

In [8]:
import json
import random

# Build final dataset
final_dataset = []

easy_examples = [p for p in parsed if p['difficulty'] == 'easy']
medium_examples = [p for p in parsed if p['difficulty'] == 'medium']
hard_examples = [p for p in parsed if p['difficulty'] == 'hard']

# Easy — use compressed thinking
for i in range(2000):
    key = f"easy_{i}"
    if key not in progress:
        continue  # skip the 7 missing ones
    
    ex = easy_examples[i]
    final_dataset.append({
        'problem': ex['problem'],
        'difficulty': 'easy',
        'source': ex['source'],
        'thinking': progress[key],
        'answer': ex['answer'],
        'original_thinking_words': ex['thinking_word_count'],
        'compressed_thinking_words': len(progress[key].split()),
        'training_text': f"<think>\n{progress[key]}\n</think>\n\n{ex['answer']}"
    })

# Medium — use compressed thinking
for i in range(2000):
    key = f"medium_{i}"
    ex = medium_examples[i]
    final_dataset.append({
        'problem': ex['problem'],
        'difficulty': 'medium',
        'source': ex['source'],
        'thinking': progress[key],
        'answer': ex['answer'],
        'original_thinking_words': ex['thinking_word_count'],
        'compressed_thinking_words': len(progress[key].split()),
        'training_text': f"<think>\n{progress[key]}\n</think>\n\n{ex['answer']}"
    })

# Hard — use ORIGINAL thinking (no compression)
for i in range(2000):
    ex = hard_examples[i]
    final_dataset.append({
        'problem': ex['problem'],
        'difficulty': 'hard',
        'source': ex['source'],
        'thinking': ex['thinking'],
        'answer': ex['answer'],
        'original_thinking_words': ex['thinking_word_count'],
        'compressed_thinking_words': ex['thinking_word_count'],  # same, no compression
        'training_text': f"<think>\n{ex['thinking']}\n</think>\n\n{ex['answer']}"
    })

# Shuffle
random.seed(42)
random.shuffle(final_dataset)

# Stats
print(f"Total examples: {len(final_dataset)}")
for diff in ['easy', 'medium', 'hard']:
    subset = [d for d in final_dataset if d['difficulty'] == diff]
    avg_orig = sum(d['original_thinking_words'] for d in subset) // len(subset)
    avg_comp = sum(d['compressed_thinking_words'] for d in subset) // len(subset)
    print(f"\n{diff.upper()}: {len(subset)} examples")
    print(f"  Original avg: {avg_orig} words")
    print(f"  Final avg:    {avg_comp} words")
    print(f"  Compression:  {avg_orig / max(avg_comp, 1):.1f}x")

# Save
with open('final_training_dataset.json', 'w') as f:
    json.dump(final_dataset, f)
print(f"\nSaved to final_training_dataset.json")

Total examples: 5993

EASY: 1993 examples
  Original avg: 964 words
  Final avg:    164 words
  Compression:  5.9x

MEDIUM: 2000 examples
  Original avg: 2789 words
  Final avg:    469 words
  Compression:  5.9x

HARD: 2000 examples
  Original avg: 7640 words
  Final avg:    7640 words
  Compression:  1.0x

Saved to final_training_dataset.json
